**KuaiRand-Pure con acción = video**

Se introduce el protocolo a utilizar, en donde, a diferencia de la versión inicial en **KuaiRand-obp-tag.ipynb** , se define la acción como el video por recomendar (7 583 acciones) en lugar del primer tag. El resto del protocolo se mantiene.

In [1]:
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import ranksums
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import KFold
from sklearn.preprocessing import MultiLabelBinarizer
from obp.ope import (OffPolicyEvaluation, DirectMethod, InverseProbabilityWeighting,
                     SelfNormalizedInverseProbabilityWeighting, DoublyRobust, SelfNormalizedDoublyRobust,
                     SwitchDoublyRobustTuning, DoublyRobustWithShrinkageTuning)

c:\Users\ddono\Desktop\2026-2\RecSys\Proyecto\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
KUAI_PATH = Path("Contenido/Datasets/KuaiRand-Pure/data")
log = pd.read_csv(KUAI_PATH / "log_random_4_22_to_5_08_pure.csv", usecols=["user_id", "video_id", "is_click", "is_rand"])
assert log["is_rand"].eq(1).all()

acciones = np.sort(log["video_id"].unique())
log["accion"] = pd.Index(acciones).get_indexer(log["video_id"])
n_acciones = len(acciones)
ctr_random = log["is_click"].mean()
print(f"{len(log):,} impresiones | {log['user_id'].nunique():,} usuarios | {n_acciones:,} videos | CTR {ctr_random:.4f}")

1,186,059 impresiones | 27,285 usuarios | 7,583 videos | CTR 0.1762


## Contexto 

Se utilizan los mismos criterios de selección y tratamiento de variables para definir el contexto a utilizar (igual que en **KuaiRand-obp-tag.ipynb**)

In [3]:
usuarios = pd.read_csv(KUAI_PATH / "user_features_pure.csv")
niveles = usuarios.drop(columns="user_id").nunique()
categoricas = [c for c in niveles.index if 2 <= niveles[c] <= 15]
contexto_usuario = pd.get_dummies(usuarios[categoricas].astype(str), drop_first=True).astype("float32")
contexto_usuario.index = usuarios["user_id"]
print(f"{len(categoricas)} columnas -> {contexto_usuario.shape[1]} dummies")

20 columnas -> 88 dummies


### Features de la acción

Con 7 583 acciones (videos), el modelo de recompensa usa features del video (action_context), tales como sus tags y su duración en segundos.

Para la interpretación de los tags, se utilizó multi-hot, similar a One-hot encoding, que permite multiplicidad de índices, dado que un video puede tener varias etiquetas.

In [4]:
vf = pd.read_csv(KUAI_PATH / "video_features_basic_pure.csv", usecols=["video_id", "tag", "video_duration"]).set_index("video_id").loc[acciones]
tags = vf["tag"].fillna("sin tag").str.split(",")
contexto_accion = np.c_[MultiLabelBinarizer().fit_transform(tags), vf["video_duration"].to_numpy() / 1000].astype("float32")
print("contexto_accion:", contexto_accion.shape, "| videos sin duración:", vf["video_duration"].isna().sum())

contexto_accion: (7583, 48) | videos sin duración: 239


### Políticas


La política de evaluación $\pi_e$ es la política aleatoria del log, es decir, cada video se elige con su frecuencia empírica $p_0$, de modo que su valor real es el CTR del log aleatorio, 0,1762. 

Por otro lado, la política de logging se define como $\pi_b \propto p_0 \cdot \exp(\alpha \cdot \widetilde{CTR})$, donde $\alpha$ se calibra para que el CTR bajo $\pi_b$ sea 1,316 veces el de Random, la misma razón entre BTS y Random de OBD men. 

Al tomar el video como acción, el CTR de cada uno se estima con menos impresiones que el de un tag, por lo que inclinar $\pi_b$ hacia el CTR crudo concentraría la probabilidad en videos con CTR alto por azar. Por esta razón $\widetilde{CTR}$ corresponde al CTR suavizado hacia la media global mediante un prior beta estimado por momentos, con una fuerza equivalente a 22,8 impresiones, lo que reduce la desviación estándar del CTR entre videos de 0,0923 a 0,0770. 

La calibración, en cambio, usa el CTR crudo, ya que este determina la recompensa esperada en el log. Así se obtiene $\alpha = 6{,}46$, con un CTR bajo $\pi_b$ de 0,2319, y $c = 11{,}8$, lo que deja en promedio 100 774 filas aceptadas por pasada, suficientes para muestrear n = 10 000 en cada semilla.

In [5]:
por_video = log.groupby("accion")["is_click"].agg(["mean", "size"])
p0 = (por_video["size"] / len(log)).to_numpy()
ctr_video = por_video["mean"].to_numpy()
k = por_video["size"].to_numpy()

valor_real = (p0 * ctr_video).sum()
print(f"valor real de pi_e = p0: {valor_real:.4f}")

valor real de pi_e = p0: 0.1762


In [6]:
varianza_muestral = (ctr_random * (1 - ctr_random) / k).mean()
fuerza_prior = ctr_random * (1 - ctr_random) / (ctr_video.var() - varianza_muestral) - 1
ctr_suavizado = (ctr_video * k + ctr_random * fuerza_prior) / (k + fuerza_prior)
print(f"fuerza del prior {fuerza_prior:.1f} | CTR crudo: DE {ctr_video.std():.4f} | CTR suavizado: DE {ctr_suavizado.std():.4f}")

fuerza del prior 22.8 | CTR crudo: DE 0.0923 | CTR suavizado: DE 0.0770


In [7]:
razon_bts = 0.00674 / 0.00512     # CTR de BTS / Random en OBD men (ver KuaiRand-obp-tag.ipynb)
objetivo = razon_bts * ctr_random


def politica_logging(alpha):
    pesos = p0 * np.exp(alpha * (ctr_suavizado - ctr_suavizado.max()))
    return pesos / pesos.sum()


alpha = brentq(lambda a: (politica_logging(a) * ctr_video).sum() - objetivo, 0, 500)
pi_b = politica_logging(alpha)
c = (pi_b / p0).max()
print(f"alpha {alpha:.2f} | CTR bajo pi_b {(pi_b * ctr_video).sum():.4f} (objetivo {objetivo:.4f}) "
      f"| c {c:.1f} | filas aceptadas esperadas por pasada {len(log) / c:,.0f}")

alpha 6.46 | CTR bajo pi_b 0.2319 (objetivo 0.2319) | c 11.8 | filas aceptadas esperadas por pasada 100,774


In [8]:
acciones_log = log["accion"].to_numpy()
clics_log = log["is_click"].to_numpy()
usuarios_log = log["user_id"].to_numpy()
prob_aceptar = (pi_b / (c * p0))[acciones_log]


def log_pi_b(semilla, n):
    """Muestreo por rechazo sobre el log aleatorio (Li et al., 2011)."""
    rng = np.random.default_rng(semilla)
    aceptadas = np.flatnonzero(rng.random(len(log)) < prob_aceptar)
    idx = rng.choice(aceptadas, size=n, replace=False)
    a = acciones_log[idx]
    return {"n_rounds": n, "n_actions": n_acciones, "action": a, "reward": clics_log[idx],
            "context": contexto_usuario.loc[usuarios_log[idx]].to_numpy(),
            "position": np.zeros(n, dtype=int), "pscore": pi_b[a]}

### Experimento

In [9]:
LAMBDAS = [1, 5, 10, 50, 100, 500, 1000, 5000, 10000]
TUNING = dict(lambdas=LAMBDAS, tuning_method="mse", delta=0.05)


def crear_estimadores():
    return [
        InverseProbabilityWeighting(estimator_name="IPW"),
        SelfNormalizedInverseProbabilityWeighting(estimator_name="SNIPW"),
        DirectMethod(estimator_name="DM"),
        DoublyRobust(estimator_name="DR"),
        SelfNormalizedDoublyRobust(estimator_name="SNDR"),
        SwitchDoublyRobustTuning(**TUNING, estimator_name="Switch-DR"),
        DoublyRobustWithShrinkageTuning(**TUNING, estimator_name="DRos"),
    ]

In [10]:
def estimar_q(contexto_log, accion, recompensa, contexto_accion, semilla, filas_por_bloque=300_000):
    """q_hat(x, a) con cross-fitting K = 2, prediciendo una vez por contexto distinto."""
    n, n_acciones = len(accion), len(contexto_accion)
    X = np.c_[contexto_log, contexto_accion[accion]]
    q = np.zeros((n, n_acciones, 1))
    for ent, prueba in KFold(n_splits=2, shuffle=True, random_state=semilla).split(X):
        modelo = HistGradientBoostingClassifier(learning_rate=0.01, max_iter=100, max_depth=5,
                                                min_samples_leaf=10, random_state=semilla)
        modelo.fit(X[ent], recompensa[ent])
        distintos, inv = np.unique(contexto_log[prueba], axis=0, return_inverse=True)
        pred = np.empty((len(distintos), n_acciones), dtype=np.float32)
        paso = max(1, filas_por_bloque // n_acciones)
        for i in range(0, len(distintos), paso):
            bloque = distintos[i:i + paso]
            Xb = np.hstack([np.repeat(bloque, n_acciones, axis=0), np.tile(contexto_accion, (len(bloque), 1))]).astype(np.float32, copy=False)
            pred[i:i + paso] = modelo.predict_proba(Xb)[:, 1].reshape(len(bloque), n_acciones)
        q[prueba, :, 0] = pred[inv.ravel()]
    return q

In [11]:
def iteracion_video(semilla, n):
    fb = log_pi_b(semilla, n)
    q_hat = estimar_q(fb["context"], fb["action"], fb["reward"], contexto_accion, semilla)
    pi_e = np.broadcast_to(p0[np.newaxis, :, np.newaxis], (n, n_acciones, 1))
    ope = OffPolicyEvaluation(bandit_feedback=fb, ope_estimators=crear_estimadores())
    est = ope.estimate_policy_values(action_dist=pi_e, estimated_rewards_by_reg_model=q_hat)
    est["Ingenuo"] = fb["reward"].mean()
    del q_hat, fb, ope
    return [{"semilla": semilla, "estimador": k, "estimado": v} for k, v in est.items()]

In [12]:
N = 10_000
N_SEMILLAS = 200
ARCHIVO = Path(f"resultados/kuairand_video_n{N}.csv")
ARCHIVO.parent.mkdir(exist_ok=True)

hechas = set(pd.read_csv(ARCHIVO)["semilla"]) if ARCHIVO.exists() else set()
pendientes = [s for s in range(N_SEMILLAS) if s not in hechas]
print(f"{len(hechas)} semillas ya guardadas en {ARCHIVO} | faltan {len(pendientes)}")

inicio = time.time()
for i, semilla in enumerate(pendientes, start=1):
    filas = pd.DataFrame(iteracion_video(semilla, N))
    filas["valor_real"] = valor_real
    filas["error2"] = (filas["estimado"] - valor_real) ** 2
    filas.to_csv(ARCHIVO, mode="a", header=not ARCHIVO.exists(), index=False)
    del filas
    gc.collect()
    minutos = (time.time() - inicio) / 60
    print(f"{len(hechas) + i}/{N_SEMILLAS} semillas | {minutos:.1f} min en esta ejecución | {minutos / i:.1f} min/semilla", flush=True)

7 semillas ya guardadas en resultados\kuairand_video_n10000.csv | faltan 193


8/200 semillas | 17.5 min en esta ejecución | 17.5 min/semilla


9/200 semillas | 34.8 min en esta ejecución | 17.4 min/semilla


10/200 semillas | 52.8 min en esta ejecución | 17.6 min/semilla


11/200 semillas | 63.5 min en esta ejecución | 15.9 min/semilla


12/200 semillas | 68.4 min en esta ejecución | 13.7 min/semilla


... (184 outputs left out)

197/200 semillas | 930.7 min en esta ejecución | 4.9 min/semilla


198/200 semillas | 935.3 min en esta ejecución | 4.9 min/semilla


199/200 semillas | 939.9 min en esta ejecución | 4.9 min/semilla


200/200 semillas | 944.5 min en esta ejecución | 4.9 min/semilla


### Resultados

In [13]:
def tabla_resultados(res, valor_real):
    """RMSE, sesgo y DE relativos al valor real (%), y Wilcoxon rank-sum contra el mejor (Tabla 6, Saito et al., 2021)."""
    g = res.groupby("estimador")["estimado"]
    tabla = pd.DataFrame({
        "RMSE rel. (%)": 100 * np.sqrt(res.groupby("estimador")["error2"].mean()) / valor_real,
        "Sesgo rel. (%)": 100 * (g.mean() - valor_real) / valor_real,
        "DE rel. (%)": 100 * g.std() / valor_real,
    }).sort_values("RMSE rel. (%)")
    err = res.pivot(index="semilla", columns="estimador", values="error2")
    mejor = tabla.index[0]
    tabla[f"p vs {mejor}"] = [np.nan if e == mejor else ranksums(err[mejor], err[e]).pvalue for e in tabla.index]
    return tabla.round(3)

In [14]:
res = pd.read_csv(ARCHIVO)
print(f"{res['semilla'].nunique()} semillas | valor real {valor_real:.4f}")
tabla_resultados(res, valor_real)

200 semillas | valor real 0.1762


,RMSE rel. (%),Sesgo rel. (%),DE rel. (%),p vs IPW
estimador,,,,
IPW,2.163,0.011,2.169,NaN
SNIPW,2.210,0.072,2.214,0.998
SNDR,2.211,0.075,2.215,0.821
DR,2.242,0.093,2.246,0.728
Switch-DR,2.242,0.093,2.246,0.728
DRos,2.242,0.104,2.245,0.718
DM,27.745,27.650,2.295,0.000
Ingenuo,31.850,31.766,2.323,0.000




Los estimadores basados en pesos (IPW, SNIPW, DR, SNDR, Switch-DR y DRos) alcanzan un RMSE relativo cercano a 2.2% sin diferencias significativas respecto de IPW (p > 0,7), y su error proviene casi por completo de la varianza, ya que ninguno supera un sesgo relativo de 0,11%. 

DM, en cambio, sobreestima el valor real en 27,7 %, cerca del 31,8 % del estimador ingenuo, lo que indica que el modelo de recompensa corrige poco el cambio de distribución entre $\pi_b$ y $\pi_e$. Frente a la versión por tag (KuaiRand-obp-tag.ipynb), donde estos mismos estimadores obtenían entre 6,7 % y 7,0 %, el RMSE relativo se reduce a cerca de un tercio, mientras que DM y el ingenuo prácticamente no cambian (28,9 % y 31,7 % por tag). Esta mejora es consistente con que $\pi_b$ por video se aleja menos de $\pi_e$, como muestra que el muestreo por rechazo acepta 100 774 filas por pasada frente a 37 025 en la versión por tag, de modo que los pesos $\pi_e/\pi_b$ son menos extremos pese a haber 7 583 acciones.

**Referencias**

Gao, C., et al. (2022). KuaiRand: An Unbiased Sequential Recommendation Dataset with Randomly Exposed Videos. CIKM 2022. arXiv:2208.08696.

Li, L., Chu, W., Langford, J., & Wang, X. (2011). Unbiased Offline Evaluation of Contextual-Bandit-Based News Article Recommendation Algorithms. WSDM 2011. https://doi.org/10.1145/1935826.1935878

Saito, Y., Aihara, S., Matsutani, M., & Narita, Y. (2021). Open Bandit Dataset and Pipeline. NeurIPS Datasets and Benchmarks Track. arXiv:2008.07146.